In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

ENDSWITH = "notebooks"

NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDSWITH):
    raise ValueError(f"Not in correct dir, expect end with {ENDSWITH}, but got {NOTEBOOK_DIR} instead")

BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

In [2]:
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [ ]:
# from src.yt_crawler.YtCrawlerClass import YtCrawler

# # Keep runtime artifacts at repo root (.data/...), not under src/notebooks/
# DATA_DIR = os.path.join(BASE_DIR, ".data")

# crawler = YtCrawler(
#     output_dir=os.path.join(DATA_DIR, "yt_crawler", "downloads"),
#     work_dir=os.path.join(DATA_DIR, "yt_crawler", "work"),
# )

# # link = "https://youtu.be/u4b4VA7luW0?si=jRCJtpqjXPxVC-uX"
# link = "https://youtu.be/aRGx6PX3Edc"

# audio = crawler.ingest(link)

In [ ]:
# audio.save_to(os.path.join(BASE_DIR, "benchmarks/separation/sources/speech/vietcettera_clean.wav"))

Audio(source_id='aRGx6PX3Edc', title='Cô gái nhỏ chinh phục những đỉnh cao lớn - Cầu thủ Huỳnh Như | Kháng Thương SS3 EP3', path='/Volumes/HP_P900/Users/tungnguyen/Programming/audio-prepare-pipeline-redo/benchmarks/separation/sources/speech/vietcettera_clean.wav', sample_rate=16000, duration_s=1867.88s, channels=1, format='wav')

In [4]:
from src.utils.AudioClass import Audio

newaudio = Audio.from_file(BASE_DIR + "/benchmarks/separation/sources/speech/weapons_2025.wav")

In [ ]:
from src.utils.display_audio import display_audio

display_audio(newaudio)

In [ ]:
newaudio.show_mel_spectrogram()

In [ ]:
from src.separation import HTDemucs, BSRoFormer, MelRoFormer, MVSepMDX23

# Swap backends here; all implement BaseSeparator.separate(Audio) -> Audio
# separator = HTDemucs(
#     device=device,
#     output_dir=os.path.join(DATA_DIR, "demucs", "out"),
#     work_dir=os.path.join(DATA_DIR, "demucs", "work"),
# )
# separator = BSRoFormer(
#     device=str(device),
#     output_dir=os.path.join(DATA_DIR, "bs_roformer", "out"),
#     work_dir=os.path.join(DATA_DIR, "bs_roformer", "work"),
# )
# separator = MelRoFormer(
#     device=str(device),
#     output_dir=os.path.join(DATA_DIR, "mel_roformer", "out"),
#     work_dir=os.path.join(DATA_DIR, "mel_roformer", "work"),
# )
# separator.load()
separator = MVSepMDX23(
    device=str(device),
    output_dir=os.path.join(DATA_DIR, "mvsep_mdx23", "out"),
    work_dir=os.path.join(DATA_DIR, "mvsep_mdx23", "work"),
    repo_dir=os.path.join(DATA_DIR, "mvsep_mdx23", "repo"),
)

cleaned_audio = separator.separate(newaudio)

In [ ]:
cleaned_audio.save_to(BASE_DIR + "/data/testing/yt_crawler/cleaned_mvsep_mdx23_audio.wav")
if hasattr(separator, "close"):
    separator.close()

In [ ]:
cleaned_audio.show_mel_spectrogram()